In [59]:
from google.colab import userdata

GROQ_API_KEY = userdata.get("GROQ_API_KEY")

In [60]:
!pip install -q openai

In [61]:
from openai import OpenAI

client = OpenAI(
    api_key=GROQ_API_KEY,
    base_url="https://api.x.ai/v1",
)

Call a Grok model

In [62]:
from google.colab import userdata
from openai import OpenAI

GROQ_API_KEY = userdata.get("GROQ_API_KEY")

client = OpenAI(
    api_key=GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1",
)

In [63]:
memory = {
    "task": "",
    "plan": "",
    "research": "",
    "code": "",
    "review": "",
    "output": "",
    "timestamp": ""
}

print("✅ Memory Initialized")

✅ Memory Initialized


In [64]:
def planner_agent(task):

    prompt = f"""
You are a Planner Agent.

Your responsibilities:

1. Understand the user's task.
2. Break it into logical steps.
3. Return ONLY a numbered execution plan.

Task:
{task}
"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "system",
                "content": "You are a Planner Agent."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.2,
        max_tokens=1500
    )

    plan = response.choices[0].message.content.strip()

    # Store in memory
    memory["task"] = task
    memory["plan"] = plan

    return plan

Create the Task

In [65]:
task = """
Student Marks

Tamil = 90
English = 85
Maths = 95
Science = 92
Social = 88

Calculate:

1. Total
2. Average
3. Percentage

Print the results.
"""

Run the Planner Agent

In [66]:
print("=" * 60)
print("PLANNER AGENT")
print("=" * 60)

plan = planner_agent(task)

print(plan)

PLANNER AGENT
1. Define the marks for each subject: Tamil = 90, English = 85, Maths = 95, Science = 92, Social = 88
2. Calculate the total marks: Total = Tamil + English + Maths + Science + Social
3. Calculate the average marks: Average = Total / 5
4. Calculate the percentage: Percentage = (Total / 500) * 100
5. Print the total marks
6. Print the average marks
7. Print the percentage marks


Research Agent

In [67]:
def research_agent(task, plan):

    prompt = f"""
You are a Research Agent.

Your responsibilities:

1. Read the user's task.
2. Read the execution plan.
3. Identify important concepts, formulas, or programming techniques.
4. Provide concise research notes that will help the Writer Agent.
5. Do NOT write Python code.

Task:
{task}

Execution Plan:
{plan}
"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "system",
                "content": "You are a Research Agent."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.2,
        max_tokens=2000
    )

    research = response.choices[0].message.content.strip()

    # Save to Memory
    memory["research"] = research

    return research

Run the Research Agent

In [68]:
print("\n" + "=" * 60)
print("RESEARCH AGENT")
print("=" * 60)

research = research_agent(task, plan)

print(research)


RESEARCH AGENT
Research Notes:

* The task involves calculating the total, average, and percentage of marks for a student in five subjects: Tamil, English, Maths, Science, and Social.
* Key concepts:
	+ Total marks: sum of marks in all subjects
	+ Average marks: total marks divided by the number of subjects
	+ Percentage marks: total marks divided by the maximum possible marks (500 in this case), then multiplied by 100
* Important formulas:
	+ Total = Tamil + English + Maths + Science + Social
	+ Average = Total / 5
	+ Percentage = (Total / 500) * 100
* The execution plan involves defining the marks for each subject, calculating the total, average, and percentage, and then printing the results.
* The maximum possible marks for each subject is assumed to be 100, and the total maximum possible marks is 500 (100 x 5 subjects). 

These notes should provide the necessary information for the Writer Agent to complete the task.


Writer Agent

In [69]:
def writer_agent(task, plan, research, feedback=""):

    prompt = f"""
You are a Senior Python Developer.

Your responsibilities:

1. Read the user task.
2. Follow the execution plan.
3. Use the research notes.
4. If reviewer feedback exists, improve the code.
5. Return ONLY executable Python code.
6. Do NOT include markdown like ```python.

User Task:
{task}

Execution Plan:
{plan}

Research Notes:
{research}

Reviewer Feedback:
{feedback}
"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "system",
                "content": "You are a Senior Python Developer."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.2,
        max_tokens=3000
    )

    code = response.choices[0].message.content.strip()

    # Remove Markdown if the model adds it
    code = code.replace("```python", "")
    code = code.replace("```", "").strip()

    # Save into Memory
    memory["code"] = code

    return code

In [70]:
print("=" * 60)
print("WRITER AGENT")
print("=" * 60)

code = writer_agent(task, plan, research)

print(code)

WRITER AGENT
tamil = 90
english = 85
maths = 95
science = 92
social = 88

total = tamil + english + maths + science + social
average = total / 5
percentage = (total / 500) * 100

print("Total Marks: ", total)
print("Average Marks: ", average)
print("Percentage Marks: ", percentage)


Reviewer Agent

In [71]:
def reviewer_agent(code):

    prompt = f"""
You are a Senior Python Code Reviewer.

Your responsibilities:

1. Review the Python code carefully.
2. Check correctness.
3. Check readability.
4. Check coding best practices.
5. Check variable names.
6. Check formatting.

If everything is correct,
reply ONLY

APPROVED

Otherwise explain what should be improved.

Python Code:

{code}
"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "system",
                "content": "You are a Senior Python Code Reviewer."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.1,
        max_tokens=1500
    )

    review = response.choices[0].message.content.strip()

    # Save into Memory
    memory["review"] = review

    return review

Test the Reviewer Agent

In [72]:
print("=" * 60)
print("REVIEWER AGENT")
print("=" * 60)

review = reviewer_agent(code)

print(review)

REVIEWER AGENT
The code provided is generally correct, but there are some improvements that can be made to enhance readability and follow best practices. 

Here are the suggestions:
- Variable names can be more descriptive. For example, instead of `tamil`, `english`, etc., consider using `tamil_marks`, `english_marks`, etc.
- The code can be made more scalable by using a dictionary or a list to store the subject marks, instead of individual variables.
- The total marks and percentage calculation can be done in a more robust way by considering the maximum possible marks for each subject.
- The code can be organized into functions to improve readability and reusability.

Here's an improved version of the code:

```python
def calculate_total(marks):
    """Calculate the total marks"""
    return sum(marks.values())

def calculate_average(total, num_subjects):
    """Calculate the average marks"""
    return total / num_subjects

def calculate_percentage(total, max_total):
    """Calculate

Writer Revision Loop

In [73]:
print("=" * 60)
print("WRITER AGENT - REVISION")
print("=" * 60)

if memory["review"].strip().upper() != "APPROVED":

    improved_code = writer_agent(
        task=memory["task"],
        plan=memory["plan"],
        research=memory["research"],
        feedback=memory["review"]
    )

    memory["code"] = improved_code

    print(improved_code)

else:
    print("Reviewer already approved the code.")

WRITER AGENT - REVISION
def calculate_total(marks):
    """Calculate the total marks"""
    return sum(marks.values())

def calculate_average(total, num_subjects):
    """Calculate the average marks"""
    return total / num_subjects

def calculate_percentage(total, max_total):
    """Calculate the percentage marks"""
    return (total / max_total) * 100

# Define the subject marks
subject_marks = {
    "Tamil": 90,
    "English": 85,
    "Maths": 95,
    "Science": 92,
    "Social": 88
}

# Define the maximum possible marks for each subject
max_marks_per_subject = 100
max_total = len(subject_marks) * max_marks_per_subject

# Calculate the total, average, and percentage marks
total = calculate_total(subject_marks)
average = calculate_average(total, len(subject_marks))
percentage = calculate_percentage(total, max_total)

# Print the results
print("Total Marks: ", total)
print("Average Marks: ", average)
print("Percentage Marks: ", percentage)


Review Again

In [79]:
print("=" * 60)
print("REVIEWER AGENT")
print("=" * 60)

review = reviewer_agent(code)

print(review)

REVIEWER AGENT
The code provided is generally correct, but there are a few improvements that can be made to enhance readability and follow best practices:

1. Variable names: The variable names are clear, but they could be more descriptive. For example, instead of `tamil`, `english`, etc., consider using `tamil_marks`, `english_marks`, etc.

2. Magic numbers: The code contains magic numbers like `5` and `500`. These numbers should be replaced with named constants to improve readability. For example, you can define `TOTAL_SUBJECTS = 5` and `MAX_MARKS = 500`.

3. Comments: The code could benefit from comments to explain what each section is doing.

4. Functions: The code is not very modular. Consider breaking it down into separate functions for calculating total, average, and percentage.

Here's an improved version of the code:

```python
# Define constants
TOTAL_SUBJECTS = 5
MAX_MARKS = 500

# Define subject marks
tamil_marks = 90
english_marks = 85
maths_marks = 95
science_marks = 92
s

Executor Agent

The Executor Agent is responsible for:

Executing the approved Python code. Capturing the output. Saving the output into the shared memory. Recording the execution timestamp.

In [75]:
from io import StringIO
import sys
from datetime import datetime

def executor_agent(code):

    # Remove Markdown if present
    code = code.replace("```python", "")
    code = code.replace("```", "")
    code = code.strip()

    old_stdout = sys.stdout
    sys.stdout = StringIO()

    try:
        exec(code)

        output = sys.stdout.getvalue()

        memory["output"] = output
        memory["timestamp"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

        print("Execution Successful")

    except Exception as e:

        output = f"Execution Error:\n{e}"

        memory["output"] = output
        memory["timestamp"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    finally:
        sys.stdout = old_stdout

    return output

Execute the Final Code

In [76]:
print("=" * 60)
print("EXECUTOR AGENT")
print("=" * 60)

result = executor_agent(memory["code"])

print(result)

EXECUTOR AGENT
Total Marks:  450
Average Marks:  90.0
Percentage Marks:  90.0



Memory After Execution

Your memory dictionary now contains:

{
    "task": "...",
    "plan": "...",
    "research": "...",
    "code": "...",
    "review": "...",
    "output": "Total: 450\nAverage: 90.00\nPercentage: 90.00%",
    "timestamp": "2026-07-24 12:35:18"
}

In [77]:
def memory_agent():

    print("=" * 70)
    print("MULTI-AGENT WORKFLOW SUMMARY")
    print("=" * 70)

    print("\n📋 USER TASK")
    print("-" * 70)
    print(memory["task"])

    print("\n📝 EXECUTION PLAN")
    print("-" * 70)
    print(memory["plan"])

    print("\n🔍 RESEARCH NOTES")
    print("-" * 70)
    print(memory["research"])

    print("\n💻 FINAL PYTHON CODE")
    print("-" * 70)
    print(memory["code"])

    print("\n🔎 REVIEW RESULT")
    print("-" * 70)
    print(memory["review"])

    print("\n⚙️ EXECUTION OUTPUT")
    print("-" * 70)
    print(memory["output"])

    print("\n🕒 EXECUTION TIME")
    print("-" * 70)
    print(memory["timestamp"])

    print("\n" + "=" * 70)
    print("PROJECT 4 COMPLETED SUCCESSFULLY")
    print("=" * 70)

Run the Memory Agent

In [78]:
print("=" * 70)
print("MEMORY AGENT")
print("=" * 70)

memory_agent()

MEMORY AGENT
MULTI-AGENT WORKFLOW SUMMARY

📋 USER TASK
----------------------------------------------------------------------

Student Marks

Tamil = 90
English = 85
Maths = 95
Science = 92
Social = 88

Calculate:

1. Total
2. Average
3. Percentage

Print the results.


📝 EXECUTION PLAN
----------------------------------------------------------------------
1. Define the marks for each subject: Tamil = 90, English = 85, Maths = 95, Science = 92, Social = 88
2. Calculate the total marks: Total = Tamil + English + Maths + Science + Social
3. Calculate the average marks: Average = Total / 5
4. Calculate the percentage: Percentage = (Total / 500) * 100
5. Print the total marks
6. Print the average marks
7. Print the percentage marks

🔍 RESEARCH NOTES
----------------------------------------------------------------------
Research Notes:

* The task involves calculating the total, average, and percentage of marks for a student in five subjects: Tamil, English, Maths, Science, and Social.
* Ke